# SHIPP — Git Folder ↔ Lakehouse Access Check

This notebook belongs **inside the SHIPP Git Folder** and verifies that Python/PySpark code can read the SHIPP Unity Catalog / Delta tables.

**Architecture**

```text
GitHub repo
   ↓
Databricks Git Folder
   ↓
this notebook (.ipynb)
   ↓
Python-capable Databricks compute
   ↓
Unity Catalog / Lakehouse
   ↓
bootcamp_students.shipp_bronze / shipp_silver / shipp_gold
```

There is **no separate Lakehouse connection string** in this notebook. Access is provided by the attached Databricks compute and the user's Unity Catalog permissions.

## 1. Confirm Python compute

Attach **Serverless** or another Python/Spark-capable compute resource before running this notebook. Do not attach a SQL Warehouse for Python cells.

In [ ]:
print("SHIPP Python compute works")

## 2. Confirm this notebook is running from the Git Folder

This shows the current workspace directory. The exact path depends on the Databricks workspace and Git Folder location.

In [ ]:
import os
print("Current notebook directory:")
print(os.getcwd())

## 3. Define the SHIPP Unity Catalog objects

In [ ]:
CATALOG = "bootcamp_students"
BRONZE_SCHEMA = "shipp_bronze"
SILVER_SCHEMA = "shipp_silver"
GOLD_SCHEMA = "shipp_gold"

LISTINGS_HISTORY = f"{CATALOG}.{BRONZE_SCHEMA}.lb_listings_history"
REQUESTS_HISTORY = f"{CATALOG}.{BRONZE_SCHEMA}.lb_requests_history"

print("Listings Bronze table:", LISTINGS_HISTORY)
print("Requests Bronze table:", REQUESTS_HISTORY)

## 4. Verify Unity Catalog access

In [ ]:
spark.sql("SELECT current_catalog() AS current_catalog, current_schema() AS current_schema").show(truncate=False)
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(truncate=False)

## 5. Read the Bronze Listing history

This is a **read-only** validation.

In [ ]:
bronze_listings = spark.table(LISTINGS_HISTORY)

print("Bronze listing history row count:", bronze_listings.count())
display(bronze_listings.limit(10))

## 6. Read the Bronze Request history

This is a **read-only** validation.

In [ ]:
bronze_requests = spark.table(REQUESTS_HISTORY)

print("Bronze request history row count:", bronze_requests.count())
display(bronze_requests.limit(10))

## 7. Verify the deterministic SHIPP demo records

This proves the notebook can read the exact Bronze entities required for Stage 5.

In [ ]:
from pyspark.sql import functions as F

demo_listing_history = (
    bronze_listings
    .filter(F.col("listing_id") == "demo-listing-001")
    .orderBy(F.col("_pg_lsn").desc(), F.col("_sort_by").desc())
)

demo_request_history = (
    bronze_requests
    .filter(F.col("request_id") == "demo-request-001")
    .orderBy(F.col("_pg_lsn").desc(), F.col("_sort_by").desc())
)

print("demo-listing-001 history")
display(demo_listing_history)

print("demo-request-001 history")
display(demo_request_history)

## PASS criteria

The Git Folder ↔ Lakehouse development path is ready when:

- Python compute runs successfully.
- `bootcamp_students` is visible.
- `shipp_bronze` is visible.
- `lb_listings_history` can be read.
- `lb_requests_history` can be read.
- `demo-listing-001` history is returned.
- `demo-request-001` history is returned.

After this passes, Stage 5 code can read Bronze and write trusted current-state tables into `bootcamp_students.shipp_silver`.